# QuantAlpha AI — Step 1: Data Pipeline + Validation Layer
**Universe:** Nifty 200
**Goal:** Fetch raw OHLCV + fundamentals, validate, cache to disk, ready for Technical/Fundamental/Sentiment engines.

Run this notebook top to bottom in Google Colab (Runtime → Run all).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/QuantAlpha/data', exist_ok=True)
os.chdir('/content/drive/MyDrive/QuantAlpha')

Mounted at /content/drive


## 1. Install dependencies

In [ ]:
!pip install yfinance pandas-ta jugaad-data nsepython --quiet
print("Dependencies installed.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 23.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
Dependencies installed.


## 2. Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import pandas_ta as ta
import time
import os
from datetime import datetime

pd.set_option('display.max_columns', 50)


## 3. Nifty 200 stock list
NSE publishes the official Nifty 200 constituent list as a CSV. We fetch it directly so the list
stays current — no manual typing of 200 tickers.


In [ ]:
NIFTY200_URL = "https://archives.nseindia.com/content/indices/ind_nifty200list.csv"

try:
    nifty200 = pd.read_csv(NIFTY200_URL)
    symbols = (nifty200['Symbol'] + ".NS").tolist()
except Exception as e:
    print("Direct NSE fetch failed (common from Colab IPs — NSE blocks some cloud ranges).")
    print("Fallback: upload ind_nifty200list.csv manually from niftyindices.com and re-run this cell.")
    symbols = []

print(f"Loaded {len(symbols)} symbols")
print(symbols[:10])


Loaded 200 symbols
['360ONE.NS', 'ABB.NS', 'APLAPOLLO.NS', 'AUBANK.NS', 'ADANIENSOL.NS', 'ADANIENT.NS', 'ADANIGREEN.NS', 'ADANIPORTS.NS', 'ADANIPOWER.NS', 'ATGL.NS']


**Note:** NSE's site sometimes blocks Colab's IP range. If the cell above shows 0 symbols:
1. Download the list manually from https://www.niftyindices.com/indices/equity/broad-based-indices/nifty-200
2. Upload the CSV to Colab (left sidebar → Files → upload)
3. Re-run: `nifty200 = pd.read_csv('/content/ind_nifty200list.csv')`


## 4. Fetch historical OHLCV data
Pulls 3 years of daily data per stock — enough for indicator calculation and later backtesting.
Runs in small batches with delay to avoid Yahoo Finance rate limiting.


In [ ]:
def fetch_ohlcv(symbol, period="3y", interval="1d", retries=3):
    for attempt in range(retries):
        try:
            df = yf.Ticker(symbol).history(period=period, interval=interval)
            if df.empty:
                return None
            df = df.reset_index()
            df['symbol'] = symbol
            return df
        except Exception as e:
            time.sleep(2)
    return None

os.makedirs('data/ohlcv', exist_ok=True)

ohlcv_data = {}
failed_symbols = []

for i, sym in enumerate(symbols):
    df = fetch_ohlcv(sym)
    if df is not None and len(df) > 50:  # basic sanity: need enough history for indicators
        ohlcv_data[sym] = df
        df.to_parquet(f"data/ohlcv/{sym.replace('.NS','')}.parquet")
    else:
        failed_symbols.append(sym)
    if i % 20 == 0:
        print(f"Progress: {i}/{len(symbols)} | OK: {len(ohlcv_data)} | Failed: {len(failed_symbols)}")
    time.sleep(0.3)  # be polite to the API

print(f"\nDone. Fetched {len(ohlcv_data)} stocks. Failed: {len(failed_symbols)}")
if failed_symbols:
    print("Failed symbols (retry later or check ticker validity):", failed_symbols[:20])


Progress: 0/200 | OK: 1 | Failed: 0
Progress: 20/200 | OK: 21 | Failed: 0
Progress: 40/200 | OK: 41 | Failed: 0
Progress: 60/200 | OK: 61 | Failed: 0
Progress: 80/200 | OK: 81 | Failed: 0
Progress: 100/200 | OK: 101 | Failed: 0
Progress: 120/200 | OK: 121 | Failed: 0
Progress: 140/200 | OK: 141 | Failed: 0
Progress: 160/200 | OK: 161 | Failed: 0
Progress: 180/200 | OK: 181 | Failed: 0

Done. Fetched 200 stocks. Failed: 0


## 5. Fetch raw fundamentals
Pulls balance sheet, income statement, cash flow statement, and key ratio fields.
These raw numbers are what we'll use to COMPUTE F-Score, Z-Score, ROCE, etc. ourselves in Step 2 —
nobody hands you these pre-calculated for Indian stocks, so we build from raw statements.


In [ ]:
def fetch_fundamentals(symbol, retries=3):
    for attempt in range(retries):
        try:
            t = yf.Ticker(symbol)
            info = t.info
            bs = t.balance_sheet
            cf = t.cashflow
            inc = t.financials
            return {
                'symbol': symbol,
                'info': info,
                'balance_sheet': bs,
                'cash_flow': cf,
                'income_statement': inc
            }
        except Exception:
            time.sleep(2)
    return None

os.makedirs('data/fundamentals', exist_ok=True)

fundamentals_data = {}
failed_fundamentals = []

for i, sym in enumerate(list(ohlcv_data.keys())):  # only for stocks where OHLCV succeeded
    f = fetch_fundamentals(sym)
    if f is not None:
        fundamentals_data[sym] = f
        # save each piece separately for easy reload
        base = sym.replace('.NS', '')
        if f['balance_sheet'] is not None and not f['balance_sheet'].empty:
            f['balance_sheet'].to_csv(f"data/fundamentals/{base}_balance_sheet.csv")
        if f['cash_flow'] is not None and not f['cash_flow'].empty:
            f['cash_flow'].to_csv(f"data/fundamentals/{base}_cash_flow.csv")
        if f['income_statement'] is not None and not f['income_statement'].empty:
            f['income_statement'].to_csv(f"data/fundamentals/{base}_income_statement.csv")
    else:
        failed_fundamentals.append(sym)
    if i % 20 == 0:
        print(f"Progress: {i}/{len(ohlcv_data)} | OK: {len(fundamentals_data)} | Failed: {len(failed_fundamentals)}")
    time.sleep(0.3)

print(f"\nDone. Fetched fundamentals for {len(fundamentals_data)} stocks.")


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/

Progress: 0/200 | OK: 1 | Failed: 0


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/

Progress: 20/200 | OK: 21 | Failed: 0


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/

...

Progress: 160/200 | OK: 161 | Failed: 0


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/

Progress: 180/200 | OK: 181 | Failed: 0


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:700: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  start = pd.Timestamp.utcnow().floor("D") - datetime.timedelta(days=365 // 2)
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/quote.py:702: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:131: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  end = pd.Timestamp.utcnow().ceil("D")
/usr/local/


Done. Fetched fundamentals for 200 stocks.


## 6. Data validation layer
Before anything gets used by an engine, it must pass basic sanity checks.
This is the single highest-leverage step for accuracy — bad inputs silently wreck every model
downstream, and it's much cheaper to catch it here than debug a weird prediction later.


In [ ]:
def validate_ohlcv(df, symbol):
    issues = []
    if df is None or df.empty:
        return ['empty_dataframe']
    if (df['Close'] <= 0).any():
        issues.append('non_positive_close_price')
    if (df['Volume'] < 0).any():
        issues.append('negative_volume')
    if df['Close'].isna().sum() > len(df) * 0.05:
        issues.append('too_many_missing_close_values')
    # detect suspicious single-day price jumps (>50%) that might be data errors (vs real splits)
    pct_change = df['Close'].pct_change().abs()
    if (pct_change > 0.5).sum() > 3:
        issues.append('multiple_suspicious_price_jumps_check_for_splits')
    date_gaps = df['Date'].diff().dt.days if 'Date' in df.columns else None
    return issues

validation_report = {}
for sym, df in ohlcv_data.items():
    issues = validate_ohlcv(df, sym)
    if issues:
        validation_report[sym] = issues

print(f"Stocks with data quality issues: {len(validation_report)} / {len(ohlcv_data)}")
for sym, issues in list(validation_report.items())[:10]:
    print(f"  {sym}: {issues}")

# Save validation report for reference
pd.DataFrame([
    {'symbol': s, 'issues': ', '.join(v)} for s, v in validation_report.items()
]).to_csv('data/validation_report.csv', index=False)


Stocks with data quality issues: 0 / 200


## 7. Compute technical indicators (self-computed, consistent formulas)
Every indicator is calculated directly from raw OHLCV using `pandas-ta` — same formula, same
parameters, applied identically across all 200 stocks. This is what makes scores comparable
stock-to-stock.


In [ ]:
def compute_technical_indicators(df):
    df = df.copy()
    df['RSI_14'] = ta.rsi(df['Close'], length=14)
    macd = ta.macd(df['Close'])
    df = pd.concat([df, macd], axis=1)
    df['ATR_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)
    bbands = ta.bbands(df['Close'], length=20)
    df = pd.concat([df, bbands], axis=1)
    df['SMA_50'] = ta.sma(df['Close'], length=50)
    df['SMA_200'] = ta.sma(df['Close'], length=200)
    df['EMA_20'] = ta.ema(df['Close'], length=20)
    df['ADX_14'] = ta.adx(df['High'], df['Low'], df['Close'], length=14)['ADX_14']
    df['VWAP'] = ta.vwap(df['High'], df['Low'], df['Close'], df['Volume'])
    df['OBV'] = ta.obv(df['Close'], df['Volume'])
    supertrend = ta.supertrend(df['High'], df['Low'], df['Close'])
    if supertrend is not None:
        df = pd.concat([df, supertrend], axis=1)
    # 52-week high/low (approx 252 trading days)
    df['52w_high'] = df['Close'].rolling(252, min_periods=50).max()
    df['52w_low'] = df['Close'].rolling(252, min_periods=50).min()
    df['pct_from_52w_low'] = (df['Close'] - df['52w_low']) / df['52w_low'] * 100
    return df

os.makedirs('data/technical', exist_ok=True)

technical_data = {}
for sym, df in ohlcv_data.items():
    try:
        enriched = compute_technical_indicators(df)
        technical_data[sym] = enriched
        enriched.to_parquet(f"data/technical/{sym.replace('.NS','')}.parquet")
    except Exception as e:
        print(f"Failed indicators for {sym}: {e}")

print(f"Computed technical indicators for {len(technical_data)} stocks.")
technical_data[list(technical_data.keys())[0]].tail(3) if technical_data else None


[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered DatetimeIndex.
[!] VWAP requires an ordered Dat

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,symbol,RSI_14,MACD_12_26_9,MACDh_12_26_9,MACDs_12_26_9,ATR_14,BBL_20_2.0_2.0,BBM_20_2.0_2.0,BBU_20_2.0_2.0,BBB_20_2.0_2.0,BBP_20_2.0_2.0,SMA_50,SMA_200,EMA_20,ADX_14,VWAP,OBV,SUPERT_7_3.0,SUPERTd_7_3.0,SUPERTl_7_3.0,SUPERTs_7_3.0,52w_high,52w_low,pct_from_52w_low
740,2026-07-01 00:00:00+05:30,1074.5,1084.900024,1061.699951,1065.900024,1496165,0.0,0.0,360ONE.NS,40.484167,-1.629889,-6.365127,4.735238,28.756584,1033.660317,1096.665015,1159.669713,11.490236,0.255852,1090.005549,1093.311838,1096.429385,13.231574,None,31015074.0,1056.136537,1.0,1056.136537,NaN,1239.464355,923.886902,15.371267
741,2026-07-02 00:00:00+05:30,1075.0,1081.599976,1060.599976,1070.000000,905046,0.0,0.0,360ONE.NS,42.015189,-3.427519,-6.530205,3.102686,28.202542,1033.415804,1096.575012,1159.734220,11.519359,0.289619,1090.482009,1093.330064,1093.912301,13.110361,None,31920120.0,1056.136537,1.0,1056.136537,NaN,1239.464355,923.886902,15.815042
742,2026-07-03 00:00:00+05:30,1084.0,1120.900024,1077.500000,1109.500000,1319185,0.0,0.0,360ONE.NS,54.230963,-1.645862,-3.798838,2.152977,29.823791,1035.883647,1098.355011,1160.826375,11.375441,0.589201,1092.000010,1093.649161,1095.396844,12.984803,None,33239305.0,1056.136537,1.0,1056.136537,NaN,1239.464355,923.886902,20.090457


## 8. Summary + next step
At this point you have, cached to disk under `data/`:
- `data/ohlcv/*.parquet` — validated raw price history per stock
- `data/fundamentals/*.csv` — raw balance sheet / cash flow / income statement per stock
- `data/technical/*.parquet` — OHLCV + full technical indicator set per stock
- `data/validation_report.csv` — data quality flags to review before trusting a stock's signals

**Next (Step 2):** compute Piotroski F-Score, Altman Z-Score, ROCE, FCF margin, etc. from the raw
fundamentals saved here, and build the "temporary vs permanent event" sentiment classifier.

**Before moving on:** download the `data/` folder (or push it to GitHub / Google Drive) so it
persists after this Colab session ends.


In [10]:
import os
print("Current directory:", os.getcwd())
print("\nFiles here:")
for f in os.listdir():
    print(" -", f)
print("\nContents of data/ folder:")
for root, dirs, files in os.walk('data'):
    print(root, "→", len(files), "files")

Current directory: /content/drive/MyDrive/QuantAlpha

Files here:
 - data
 - quantalpha_step1_data.zip

Contents of data/ folder:
data → 1 files
data/ohlcv → 200 files
data/fundamentals → 600 files
data/technical → 200 files


In [ ]:
import shutil
shutil.make_archive('quantalpha_step1_data', 'zip', 'data')
print("Zipped to quantalpha_step1_data.zip — download this from the Colab file browser.")


Zipped to quantalpha_step1_data.zip — download this from the Colab file browser.
